# Evon on Kaggle, free

Serves **Gnani Evon v3.3** (30B-A3B, IQ4_XS GGUF) with llama.cpp on Kaggle's free **2× T4** and exposes it over a free Cloudflare tunnel, so the Nyayamitra API can call it.

**Settings before you run:** Accelerator → **GPU T4 x2**, Internet → **On** (needs a phone-verified Kaggle account). Then *Run all*.

Kaggle gives 30 GPU hours a week, 12 hours per session. The tunnel URL changes every session; the last cell prints the two lines to paste.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Build llama.cpp with CUDA (≈8 min)

In [ ]:
%%bash
set -e
cd /tmp
[ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp
cd llama.cpp
cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF > /dev/null
cmake --build build --config Release -j 4 --target llama-server 2>&1 | tail -n 2
ls -lh build/bin/llama-server

## 2 · Download Evon (18 GB, ≈5 min)

In [ ]:
!pip -q install -U "huggingface_hub[hf_transfer]"
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

MODEL = hf_hub_download(
    "luminenio/gnani-evon-v3.3-30B-A3B-GGUF",
    "gnani-evon-v3.3-30B-A3B-IQ4_XS.gguf",
    local_dir="/tmp/models",
)
print(MODEL)

## 3 · Start the server
A random key protects it, so only your API can use the tunnel.

In [ ]:
import secrets, subprocess, time, urllib.request

API_KEY = secrets.token_urlsafe(24)
server = subprocess.Popen(
    ["/tmp/llama.cpp/build/bin/llama-server", "-m", MODEL, "--alias", "evon",
     "-ngl", "999", "--split-mode", "layer", "--tensor-split", "1,1",
     "-c", "16384", "--parallel", "2", "--jinja",
     "--host", "0.0.0.0", "--port", "8080", "--api-key", API_KEY],
    stdout=open("/tmp/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(180):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).status == 200:
            print("Evon is up"); break
    except Exception:
        time.sleep(2)
else:
    print(open("/tmp/server.log").read()[-3000:])

## 4 · Quick test in Hindi

In [ ]:
import json, urllib.request
req = urllib.request.Request(
    "http://127.0.0.1:8080/v1/chat/completions",
    data=json.dumps({
        "model": "evon",
        "messages": [{"role": "user", "content": "एक वाक्य में बताइए: अग्रिम ज़मानत क्या है?"}],
        "max_tokens": 120,
        "chat_template_kwargs": {"enable_thinking": False},
    }).encode(),
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"},
)
t = time.time()
print(json.load(urllib.request.urlopen(req))["choices"][0]["message"]["content"])
print(f"{time.time() - t:.1f}s")

## 5 · Open the tunnel and copy the lines it prints

In [ ]:
import re
!wget -q -nc -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0); break
print("Run this on your Mac (from the nyayamitra folder):\n")
print(f"./deploy/evon-url.sh {URL}/v1 {API_KEY}")

## 6 · Keep it alive
Leave this running. Stop it (or the session) when you are done to save GPU hours.

In [ ]:
while True:
    ok = server.poll() is None and tunnel.poll() is None
    print(time.strftime("%H:%M"), "serving" if ok else "STOPPED, rerun cells 3 and 5", flush=True)
    time.sleep(300)